In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from google import genai
from google.genai.types import HttpOptions

load_dotenv()

PROJECT_ID = "llm-zoomcamp-500505"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)

In [9]:
import json
user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=data_gen_instructions,
        response_mime_type="application/json",
        response_schema=Questions,
    ),
)

In [13]:
response.parsed.questions
result = Questions.model_validate_json(response.text)
result.questions

['I just found out about the course, can I still enroll?',
 'Is it possible to get a completion certificate for this course?',
 "What's required to earn a certificate for the course?",
 'Is there a specific timeframe for submitting projects if I want a certificate?',
 'Even if I join later in the course, can I still qualify for a certificate?']

In [14]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [15]:
from evaluation_utils import llm_structured

In [16]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found the course, is it too late to join and participate?', 'What are the requirements to receive a certificate for this course?', 'If I enroll late, am I still eligible to earn a completion certificate?', 'Is there a deadline for project submissions to qualify for a certificate?', 'Does completing and submitting a project affect whether I get a certificate?']


In [17]:
usage

GenerateContentResponseUsageMetadata(
  candidates_token_count=95,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=95
    ),
  ],
  prompt_token_count=179,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=179
    ),
  ],
  thoughts_token_count=844,
  total_token_count=1118,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
)

In [18]:
from evaluation_utils import calc_price

In [19]:
print(usage)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=95 candidates_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=95
)] prompt_token_count=179 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=179
)] thoughts_token_count=844 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=1118 traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>


In [20]:
import evaluation_utils

print(evaluation_utils.__file__)

/workspaces/llm-zoomcamp-2026-code/04-evaluation/code/evaluation_utils.py


In [21]:
import inspect
import evaluation_utils

print(inspect.getsource(evaluation_utils.calc_price))

def calc_price(usage):
    input_tokens = usage.prompt_token_count or 0
    output_tokens = usage.candidates_token_count or 0

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": usage.total_token_count,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
    }



In [22]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

<module 'evaluation_utils' from '/workspaces/llm-zoomcamp-2026-code/04-evaluation/code/evaluation_utils.py'>

In [23]:
cost = evaluation_utils.calc_price(usage)
print(cost)

{'input_tokens': 179, 'output_tokens': 95, 'total_tokens': 1118, 'input_cost': 0.00013424999999999998, 'output_cost': 0.00042750000000000004, 'total_cost': 0.00056175}


In [24]:
# cost = calc_price(usage)

print(cost)

{'input_tokens': 179, 'output_tokens': 95, 'total_tokens': 1118, 'input_cost': 0.00013424999999999998, 'output_cost': 0.00042750000000000004, 'total_cost': 0.00056175}


In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found the course, is it too late to join and participate?',
  'document': '74eb249bbf'},
 {'question': 'What are the requirements to receive a certificate for this course?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll late, am I still eligible to earn a completion certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for project submissions to qualify for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Does completing and submitting a project affect whether I get a certificate?',
  'document': '74eb249bbf'}]

In [26]:
import pandas as pd

In [27]:
pd.DataFrame(records)

,question,document
0,"I just found the course, is it too late to joi...",74eb249bbf
1,What are the requirements to receive a certifi...,74eb249bbf
2,"If I enroll late, am I still eligible to earn ...",74eb249bbf
3,Is there a deadline for project submissions to...,74eb249bbf
4,Does completing and submitting a project affec...,74eb249bbf


In [28]:
from evaluation_utils import llm_structured_retry

In [29]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [30]:
generate_ground_truth(doc)

([{'question': "Can I still enroll in the course even if I'm a bit late?",
   'document': '74eb249bbf'},
  {'question': 'What are the requirements if I want to receive a certificate for this course?',
   'document': '74eb249bbf'},
  {'question': 'Is there a deadline for submitting projects to get the certificate?',
   'document': '74eb249bbf'},
  {'question': "Do I need to submit a project even if I'm not interested in getting a certificate?",
   'document': '74eb249bbf'},
  {'question': 'If I join the course late, can I still qualify for a certificate?',
   'document': '74eb249bbf'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=104,
   candidates_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=104
     ),
   ],
   prompt_token_count=179,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=179
     ),
   ],
   thoughts_token_count=887,
   tota

In [31]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [32]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [33]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

NameError: name 'results' is not defined

In [ ]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.05718675000000001

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.05718675000000001

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)

395